# Gemma 3 4B on many more Open Pool rows (Unsloth QLoRA)

**A training run, not a submission.** The 18,000-row Gemma vs Qwen run could not separate the two families (cross-family minus same-family: +0.002, interval -0.009 to +0.012). Gemma trained faster (65 vs 79 minutes). The n-gram baseline gained a lot from more rows (0.311 on 2,160 rows, 0.398 on 18,000). So this run gives Gemma 3 4B far more rows.

**What changed from the Gemma vs Qwen notebook**
- **Gemma only, 4,000 rows per model (72,000 rows), split between two runs.** The runs train side by side, one per GPU, each on its own half (2,000 rows per model, no overlap). That is about 1,125 steps per run at roughly 6.9 s per step, so about 2.2 hours, plus an n-gram fit on all 72,000 rows first (not measured). Set `SPLIT_ROWS = False` for both runs to train on all rows, which takes about twice as long.
- **Equal rows per model.** If any model has fewer than `PER_CLASS` usable pool rows, every model is cut to the smallest count. `prepare.json` records how many rows each model had and what was used.
- **Saves what an offline submission notebook needs.** Scoring has no internet. The last cell copies the base model and the packages the install step added into `/kaggle/working/offline`. It is best effort, and the results are already saved before it runs. You do not need to download `offline/`: in a submission notebook, add this notebook's output as an input on Kaggle.
- Everything that made the last version crash-safe is unchanged: separate training process, checkpoints about every 10 minutes, finished runs skipped, rerun the last cells to resume.

**How to run it**
1. Add the **Name That LLM** competition data and the [Name That LLM Open Pool](https://www.kaggle.com/datasets/crtproduction/name-that-llm-open-pool) as inputs.
2. Choose **GPU T4 x2** and turn **Internet on**.
3. Leave `SMOKE = True` and use *Run All* (about 15 minutes, plus the copy in the last cell). The job-to-GPU assignment and the last cell are new, so do not skip this.
4. Set `SMOKE = False` and use **Save Version > Save & Run All**. Expect roughly 3 hours.

Memory and GPU use are printed with every progress line, so if a crash does happen, the last lines show whether it ran out of memory.


## 1. Settings

In [ ]:
SMOKE = True   # set False for the real run

FAMILIES = {
    'gemma': 'unsloth/gemma-3-4b-pt-unsloth-bnb-4bit',     # 4.4 GB
}
UNSLOTH_VERSION = '2026.11.3'   # the version these scripts were tested with
SEEDS = [1, 2]                  # one run per seed, each on its own GPU
SPLIT_ROWS = True               # each seed trains on its own slice of the rows (half each), so a run takes half as long and the runs do not repeat each other
PER_CLASS = 40 if SMOKE else 4000     # training rows per model: 72,000 rows in total if every model has that many usable rows
BUDGET_MIN = 10 if SMOKE else 180     # training time limit for each run (1,125 steps at ~6.9 s is about 130 minutes)
TOTAL_HOURS = 1 if SMOKE else 7       # hard stop for everything
LIMIT_STEPS = 12 if SMOKE else 0
EVAL_ROWS = 540 if SMOKE else 0       # 0 means every official row


## 2. Scripts

In [ ]:
%%writefile check_env.py
"""Import the training stack exactly as the workers will, so an install problem shows up in minutes."""
import importlib.metadata, importlib.util


def _torchao_too_old():
    try:
        version = importlib.metadata.version('torchao')
    except importlib.metadata.PackageNotFoundError:
        return False
    from packaging.version import parse
    return parse(version) < parse('0.16.0')


hidden = _torchao_too_old()
if hidden:
    _find_spec = importlib.util.find_spec
    importlib.util.find_spec = lambda name, *args, **kw: None if name.split('.')[0] == 'torchao' else _find_spec(name, *args, **kw)
import unsloth
from unsloth import FastModel  # noqa: F401
import bitsandbytes, peft, torch, transformers
print('unsloth', unsloth.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__, '| bitsandbytes', bitsandbytes.__version__)
print('torchao hidden from peft (too old):', hidden)
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


In [ ]:
%%writefile prepare.py
"""Build the training rows from the Open Pool and keep them apart from the official rows."""
import argparse, json, re, unicodedata, zlib
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

ap = argparse.ArgumentParser()
ap.add_argument('--data', required=True); ap.add_argument('--pool', required=True); ap.add_argument('--work', required=True)
ap.add_argument('--per-class', type=int, default=2000); ap.add_argument('--seed', type=int, default=0)
a = ap.parse_args()
work = Path(a.work); work.mkdir(parents=True, exist_ok=True)

official = pd.read_csv(Path(a.data) / 'train.csv'); official['text'] = official.text.fillna('').astype(str)
labels = sorted(official.model.unique()); assert len(labels) == 18, labels

FLAGS = ['flag_train_prompt_regen', 'flag_near_train_prompt', 'flag_gemini37_routing_risk', 'flag_kb_proxy']
pool = pd.read_parquet(a.pool, columns=['text', 'model', 'match_weight'] + FLAGS)
audit = {'pool_rows': int(len(pool))}
pool = pool[pool.model.isin(labels)].copy()
pool['text'] = pool.text.fillna('').astype(str).str.strip()   # line breaks stay: they are part of an author's style
keep = pool.text.str.len() > 0
for f in FLAGS:
    hit = pool[f].fillna(False).astype(bool); audit[f] = int(hit.sum()); keep &= ~hit
pool = pool[keep]

def norm(t): return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', t)).strip()
exact = pool.text.map(norm).isin(set(official.text.map(norm)))
audit['exact_official_matches_removed'] = int(exact.sum())
pool = pool[~exact].drop_duplicates(['text', 'model'])

rng = np.random.default_rng(a.seed); parts = []
counts = pool.model.value_counts().reindex(labels).fillna(0).astype(int)
per = min(a.per_class, int(counts.min()))   # every model gets the same number of rows
audit['usable_rows_per_model'] = {k: int(v) for k, v in counts.items()}; audit['per_class_used'] = per
if per < 30: raise SystemExit(f'Only {int(counts.min())} usable pool rows for {counts.idxmin()}: check the Open Pool version')
for lab in labels:   # favouring rows whose typography matches the official rows
    g = pool[pool.model == lab]; n = per
    w = pd.to_numeric(g.match_weight, errors='coerce').fillna(0).clip(lower=0).to_numpy(float)
    keys = np.log(np.maximum(w, 1e-12)) + rng.gumbel(size=len(g))
    parts.append(g.iloc[np.argsort(-keys)[:n]][['text', 'model']])
train = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=a.seed).reset_index(drop=True)
train.to_parquet(work / 'train_pool.parquet', index=False)
audit['train_rows'] = int(len(train)); audit['rows_per_model'] = {k: int(v) for k, v in train.model.value_counts().items()}

def shingles(t, n=8):
    w = re.findall(r'\w+', t.lower()); return {zlib.crc32(' '.join(w[i:i + n]).encode()) for i in range(len(w) - n + 1)}
seen = set()
for t in train.text: seen |= shingles(t)
share = np.array([len(s & seen) / max(len(s), 1) for s in map(shingles, official.text)])
audit['official_rows_sharing_20pct_of_8grams'] = float((share > 0.2).mean())
audit['mean_8gram_overlap'] = float(share.mean())
# The near-prompt flags can be empty, so also measure directly whether the training rows cover the official prompts.
# 8-grams miss this: two answers to one prompt share a topic but few exact phrases.
vec = TfidfVectorizer(sublinear_tf=True, stop_words='english', min_df=2, max_df=0.05)
X = vec.fit_transform(pd.concat([official.text, train.text])); A, B = X[:len(official)], X[len(official):]
nearest = np.concatenate([(A[s:s + 256] @ B.T).max(axis=1).toarray().ravel() for s in range(0, A.shape[0], 256)])
sim = (A @ A.T).toarray(); np.fill_diagonal(sim, np.nan); same = official.prompt_id.to_numpy()[:, None] == official.prompt_id.to_numpy()[None, :]
same_prompt = np.nanmean(np.where(same, sim, np.nan), axis=1)   # how alike two answers to one prompt usually are
other_prompt = np.nanmax(np.where(same, np.nan, sim), axis=1)   # the closest answer to a different prompt
audit['topic_similarity_same_prompt_median'] = float(np.nanmedian(same_prompt))
audit['topic_similarity_nearest_other_prompt_median'] = float(np.nanmedian(other_prompt))
audit['topic_similarity_nearest_training_row_median'] = float(np.median(nearest))
audit['official_rows_on_a_prompt_the_training_rows_cover'] = float(np.nanmean(nearest > same_prompt))
(work / 'prepare.json').write_text(json.dumps(audit, indent=1))
print(json.dumps({k: v for k, v in audit.items() if k != 'rows_per_model'}, indent=1))
if audit['official_rows_sharing_20pct_of_8grams'] > 0.01:
    print('WARNING: over 1% of official rows closely overlap the training rows, so absolute scores may be inflated.')
if audit['official_rows_on_a_prompt_the_training_rows_cover'] > 0.10:
    print('WARNING: the training rows appear to cover many official prompts, so absolute scores may be inflated. The Qwen vs Gemma comparison is less affected.')


In [ ]:
%%writefile ngram.py
"""Style baseline: an n-gram SVC trained on the same pool rows, scored on the official rows."""
import argparse, json
from pathlib import Path
import numpy as np, pandas as pd
from scipy.special import log_softmax
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

ap = argparse.ArgumentParser()
ap.add_argument('--data', required=True); ap.add_argument('--work', required=True); ap.add_argument('--eval-rows', type=int, default=0)
a = ap.parse_args()
work = Path(a.work); train = pd.read_parquet(work / 'train_pool.parquet')
official = pd.read_csv(Path(a.data) / 'train.csv'); official['text'] = official.text.fillna('').astype(str)
if a.eval_rows: official = official.head(a.eval_rows)
labels = sorted(pd.read_csv(Path(a.data) / 'train.csv').model.unique())
svc = Pipeline([('f', FeatureUnion([
    ('char', TfidfVectorizer(analyzer='char', ngram_range=(2, 5), max_features=300000, sublinear_tf=True, lowercase=False, dtype=np.float32)),
    ('word', TfidfVectorizer(ngram_range=(1, 2), max_features=100000, sublinear_tf=True, lowercase=False, token_pattern=r'\w+|[^\w\s]', dtype=np.float32))])),
    ('clf', LinearSVC(C=0.5, class_weight='balanced', max_iter=5000))])
svc.fit(train.text, train.model)
d = svc.decision_function(official.text)[:, [list(svc.classes_).index(l) for l in labels]]
np.savez(work / 'ngram.npz', logp=log_softmax(d * 4, axis=1).astype(np.float32))
print('ngram done', d.shape, flush=True)


In [ ]:
%%writefile worker_unsloth.py
"""Train one Unsloth QLoRA classifier on the pool rows, then score the official rows, which it never trains on.
Saves a checkpoint every few minutes and resumes from it, so a crash costs minutes, not the whole run."""
import argparse, importlib.metadata, importlib.util, json, math, os, resource, time
from pathlib import Path


def _torchao_too_old():
    try:
        version = importlib.metadata.version('torchao')
    except importlib.metadata.PackageNotFoundError:
        return False
    from packaging.version import parse
    return parse(version) < parse('0.16.0')


if _torchao_too_old():   # peft refuses old torchao builds. Nothing here uses torchao, so hide it before peft looks.
    _find_spec = importlib.util.find_spec
    importlib.util.find_spec = lambda name, *args, **kw: None if name.split('.')[0] == 'torchao' else _find_spec(name, *args, **kw)

ap = argparse.ArgumentParser()
ap.add_argument('--name', required=True); ap.add_argument('--repo', required=True); ap.add_argument('--dtype', choices=['fp16', 'bf16'], default='fp16')
ap.add_argument('--seed', type=int, required=True); ap.add_argument('--data', required=True); ap.add_argument('--work', required=True)
ap.add_argument('--budget-min', type=float, default=120); ap.add_argument('--deadline', type=float, default=0)
ap.add_argument('--bs', type=int, default=8); ap.add_argument('--accum', type=int, default=4); ap.add_argument('--max-len', type=int, default=384)
ap.add_argument('--lr', type=float, default=1e-4); ap.add_argument('--rank', type=int, default=16)
ap.add_argument('--shard-index', type=int, default=0); ap.add_argument('--shard-count', type=int, default=1)
ap.add_argument('--limit-steps', type=int, default=0); ap.add_argument('--eval-rows', type=int, default=0); ap.add_argument('--ckpt-every', type=int, default=100)
a = ap.parse_args()

import unsloth  # noqa: F401  (must load before transformers and peft)
from unsloth import FastModel
import numpy as np, pandas as pd, torch
from torch import nn
from torch.nn import functional as F

T0 = time.time()
def log(*x): print(f'[{(time.time() - T0) / 60:7.1f}m {a.name}]', *x, flush=True)
def memory():
    free = None
    try:
        free = next(int(l.split()[1]) for l in open('/proc/meminfo') if l.startswith('MemAvailable')) / 2**20
    except (OSError, StopIteration):
        pass
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20
    gpu = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0
    return f'RAM free {free:.1f}GB, process peak {rss:.1f}GB, GPU peak {gpu:.1f}GB' if free is not None else f'process peak {rss:.1f}GB, GPU peak {gpu:.1f}GB'

out = Path(a.work) / 'runs' / a.name; out.mkdir(parents=True, exist_ok=True)
ckpt_path = out / 'ckpt.pt'
status = json.loads((out / 'run.json').read_text()) if (out / 'run.json').exists() else {}
status.update(name=a.name, repo=a.repo, seed=a.seed, status='running')
def save_status(**kw):
    status.update(kw); tmp = out / 'run.json.tmp'; tmp.write_text(json.dumps(status, indent=1)); os.replace(tmp, out / 'run.json')
def fail(reason):
    save_status(status='failed', reason=reason); raise SystemExit(f'{a.name}: {reason}')
save_status()

torch.manual_seed(a.seed); rng = np.random.default_rng(a.seed)
dev = 'cuda'
compute = torch.float16 if a.dtype == 'fp16' else torch.bfloat16
train = pd.read_parquet(Path(a.work) / 'train_pool.parquet')
if a.shard_count > 1:   # this run trains on its own slice: every model keeps the same share, so the slices do not overlap
    train = train[train.groupby('model').cumcount() % a.shard_count == a.shard_index].reset_index(drop=True)
official = pd.read_csv(Path(a.data) / 'train.csv'); official['text'] = official.text.fillna('').astype(str)
labels = sorted(official.model.unique()); to_id = {l: i for i, l in enumerate(labels)}
if a.eval_rows: official = official.head(a.eval_rows)
y = train.model.map(to_id).to_numpy()

model, processor = FastModel.from_pretrained(model_name=a.repo, max_seq_length=a.max_len, dtype=compute, load_in_4bit=True,
                                             full_finetuning=False, use_gradient_checkpointing='unsloth', random_state=a.seed)
tok = getattr(processor, 'tokenizer', processor)   # multimodal models return a processor
if tok.pad_token is None: tok.pad_token = tok.eos_token
tok.padding_side = 'right'
model = FastModel.get_peft_model(model, r=a.rank, lora_alpha=2 * a.rank, lora_dropout=0, bias='none', random_state=a.seed,
                                 finetune_vision_layers=False, finetune_language_layers=True,
                                 finetune_attention_modules=True, finetune_mlp_modules=True, use_gradient_checkpointing='unsloth')
log('loaded', memory())

def find_decoder(m):
    """The text decoder stack: the largest module with .layers and .embed_tokens outside any vision or audio tower."""
    best = None
    for name, mod in m.named_modules():
        if any(w in name.lower() for w in ('vision', 'audio', 'visual')): continue
        layers = getattr(mod, 'layers', None)
        if isinstance(layers, nn.ModuleList) and hasattr(mod, 'embed_tokens') and (best is None or len(layers) > len(best[1].layers)):
            best = (name, mod)
    if best is None: fail('could not find the text decoder inside the model')
    return best
dec_name, decoder = find_decoder(model)
n_layers = len(decoder.layers); picks = sorted({n_layers // 2, 3 * n_layers // 4, n_layers})
hidden = int(getattr(getattr(model.config, 'text_config', model.config), 'hidden_size'))
log('decoder', dec_name or '(root)', n_layers, 'layers, pooling', picks)

class Head(nn.Module):
    def __init__(self):
        super().__init__(); self.norms = nn.ModuleList(nn.LayerNorm(hidden) for _ in picks); self.out = nn.Linear(hidden * len(picks), 18)
    def features(self, pooled): return torch.cat([n(p) for n, p in zip(self.norms, pooled)], -1)
    def forward(self, pooled): return self.out(self.features(pooled))
head = Head().to(dev)

enc = tok(train.text.tolist(), truncation=True, max_length=a.max_len)['input_ids']
enc_off = tok(official.text.tolist(), truncation=True, max_length=a.max_len)['input_ids']
lens = np.array([len(x) for x in enc]); log('rows', len(enc), 'mean tokens', round(float(lens.mean())))

def batch(rows, ids):
    return tok.pad({'input_ids': [ids[int(i)] for i in rows]}, return_tensors='pt').to(dev)
def pool_layers(b):
    """Mean over real tokens of a few layers, skipping position 0. The layers come from the decoder's own hidden_states:
    reading them with forward hooks would capture tensors that Unsloth's checkpointing detaches, so LoRA would get no gradient."""
    with torch.autocast(dev, dtype=compute):
        hs = decoder(input_ids=b['input_ids'], attention_mask=b['attention_mask'], output_hidden_states=True, use_cache=False).hidden_states
    if hs is None or len(hs) != n_layers + 1: fail(f'the decoder returned {None if hs is None else len(hs)} hidden states, expected {n_layers + 1}')
    m = b['attention_mask'].clone().float(); m[:, 0] = 0; m = m.unsqueeze(-1); den = m.sum(1).clamp(min=1)
    return [(hs[l].float() * m).sum(1) / den for l in picks]

model.eval()
with torch.inference_mode():   # fail in seconds, not hours, if this precision overflows on real passages
    probe = pool_layers(batch(np.argsort(-lens)[:8], enc))
if not all(torch.isfinite(p).all() for p in probe): fail(f'non-finite activations in {a.dtype}')
FastModel.for_training(model); head.train()

named = [(f'lora.{k}', p) for k, p in model.named_parameters() if p.requires_grad] + [(f'head.{k}', p) for k, p in head.named_parameters()]
for _, p in named:   # adapters must be fp32 for loss scaling to work
    if p.dtype != torch.float32: p.data = p.data.float()
lora = [p for k, p in named if k.startswith('lora.')]
log('trainable', sum(p.numel() for p in lora), 'LoRA tensors', len(lora))
opt = torch.optim.AdamW([{'params': lora, 'lr': a.lr, 'weight_decay': 0.0}, {'params': list(head.parameters()), 'lr': a.lr * 10, 'weight_decay': 0.01}])
peak = [g['lr'] for g in opt.param_groups]
scaler = torch.amp.GradScaler('cuda', enabled=a.dtype == 'fp16')

order = rng.permutation(len(enc)); micro = []
for s in range(0, len(order), a.bs * 64):   # group similar lengths so little compute goes on padding
    chunk = order[s:s + a.bs * 64]; chunk = chunk[np.argsort(lens[chunk], kind='stable')]
    micro += [chunk[i:i + a.bs] for i in range(0, len(chunk), a.bs)]
micro = [micro[i] for i in rng.permutation(len(micro))]
total = math.ceil(len(micro) / a.accum)
if a.limit_steps: total = min(total, a.limit_steps)
step = seen = start_micro = 0; per = None; trained_sec = 0.0; recent = []; resumed = False
start_norms = [float(p.detach().float().norm()) for p in lora]

if ckpt_path.exists():   # resume where the last attempt stopped
    st = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    params = dict(named)
    for k, t in st['trainable'].items(): params[k].data.copy_(t.to(dev))
    opt.load_state_dict(st['opt']); scaler.load_state_dict(st['scaler'])
    step, seen, start_micro, total, per, trained_sec, recent = st['step'], st['seen'], st['micro'], st['total'], st['per'], st['trained_sec'], st['recent']
    log(f'resumed at step {step} of {total} ({seen} rows done)'); save_status(resumed_at_step=step); resumed = True

def save_ckpt(next_micro):
    st = {'trainable': {k: p.detach().cpu() for k, p in named}, 'opt': opt.state_dict(), 'scaler': scaler.state_dict(), 'step': step, 'seen': seen,
          'micro': next_micro, 'total': total, 'per': per, 'trained_sec': trained_sec, 'recent': recent[-200:]}
    tmp = out / 'ckpt.pt.tmp'; torch.save(st, tmp); os.replace(tmp, ckpt_path)

def budget_left():
    left = a.budget_min * 60 - trained_sec
    if a.deadline: left = min(left, a.deadline - time.time() - 600)   # keep 10 minutes for scoring and the report
    return left
if budget_left() <= 0 and step < total: fail('no time left')
if per and step < total:   # a resumed run re-fits its schedule to the time that is left
    total = min(total, step + max(1, int(budget_left() / per)))

def set_lr():
    warm = max(1, int(0.03 * total))
    mult = (step + 1) / warm if step < warm else 0.1 + 0.45 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
    for g, p in zip(opt.param_groups, peak): g['lr'] = p * mult

TEST_CRASH = int(os.environ.get('NTL_TEST_CRASH_AT_STEP', '0'))   # used only by the local tests
checked_grad = False; t_last = time.time(); t_measure = None; loss_sum = 0.0; bad = 0
for i in range(start_micro, len(micro)):
    if step >= total: break
    rows = micro[i]; b = batch(rows, enc); target = torch.as_tensor(y[rows], device=dev)
    loss = F.cross_entropy(head(pool_layers(b)), target)
    if not torch.isfinite(loss):
        bad += 1
        if bad >= 10: fail('loss went non-finite')
        continue
    scaler.scale(loss / a.accum).backward(); seen += len(rows); loss_sum += loss.item()
    if not checked_grad:   # the silent failure mode: the head learns but LoRA never gets a gradient
        # count NaN or inf as present: loss scaling makes early fp16 gradients overflow, and the scaler then skips that step
        reached = sum(1 for p in lora if p.grad is not None and bool(p.grad.detach().ne(0).any()))
        finite = all(bool(torch.isfinite(p.grad).all()) for p in lora if p.grad is not None)
        if reached == 0: fail('LoRA received no gradient: the hidden states are detached from the adapters')
        log(f'gradient check passed: {reached} of {len(lora)} LoRA tensors reached ({"finite" if finite else "overflowed, the scaler will adjust"})'); checked_grad = True
    if (i + 1) % a.accum: continue
    scaler.unscale_(opt); nn.utils.clip_grad_norm_([p for _, p in named], 1.0)
    set_lr(); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
    recent.append(loss_sum / a.accum); loss_sum = 0.0
    now = time.time(); trained_sec += now - t_last; t_last = now
    if step == min(10, total) and not resumed:   # the adapters must actually move, not just receive gradients that every step skips
        changed = sum(1 for p, n0 in zip(lora, start_norms) if abs(float(p.detach().float().norm()) - n0) > 1e-7)
        if changed == 0: fail(f'LoRA weights unchanged after {step} steps (loss scale {scaler.get_scale():.0f}): the adapters are not training')
        log(f'training check passed: {changed} of {len(lora)} LoRA tensors changed in {step} steps')
    if step == 3 and per is None: t_measure = now; s_measure = step
    if per is None and t_measure and step == s_measure + 20 and not a.limit_steps:   # fit the schedule to the budget so the learning rate still decays
        per = (now - t_measure) / 20; fit = step + int(budget_left() / per)
        if fit < total: log(f'budget allows {fit} of {total} steps'); total = fit
        log(f'{per:.2f}s per step; about {(total - step) * per / 60:.0f} minutes left to train')
    if step % 50 == 0: log(f'step {step}/{total} rows {seen} loss {np.mean(recent[-50:]):.3f} scale {scaler.get_scale():.0f} | {memory()}'); save_status(step=step, planned_steps=total, rows_seen=seen)
    if step % a.ckpt_every == 0: save_ckpt(i + 1)
    if TEST_CRASH and step == TEST_CRASH: log('test crash'); os._exit(137)
save_ckpt(len(micro))   # a crash while scoring then resumes straight to scoring
train_min = trained_sec / 60

model.eval(); head.eval(); N = len(enc_off); logp = np.zeros((N, 18), np.float32); feats = None
with torch.inference_mode():
    idx = np.argsort([len(x) for x in enc_off], kind='stable')
    for s in range(0, N, 16):
        rows = idx[s:s + 16]; f = head.features(pool_layers(batch(rows, enc_off)))
        logp[rows] = torch.log_softmax(head.out(f), -1).cpu().numpy()
        if feats is None: feats = np.zeros((N, f.shape[1]), np.float16)
        feats[rows] = f.clamp(-60000, 60000).cpu().numpy()
if not np.isfinite(logp).all(): fail('non-finite predictions')
np.savez(out / 'pred.npz', logp=logp, feats=feats)
model.save_pretrained(out / 'adapter'); torch.save(head.state_dict(), out / 'head.pt')
true = official.model.map(to_id).to_numpy(); pred = logp.argmax(1)
score = float(np.mean([(pred[true == k] == k).mean() for k in range(18) if (true == k).any()]))
save_status(status='done', step=step, planned_steps=total, rows_seen=seen, train_minutes=round(train_min, 1),
            final_loss=round(float(np.mean(recent[-50:])), 4) if recent else None, official_score=round(score, 4), peak=memory())
ckpt_path.unlink(missing_ok=True)
log('done', json.dumps(status))


In [ ]:
%%writefile orchestrate.py
"""Run every job to completion. It runs in its own session, so it keeps going if the notebook kernel dies.
Finished jobs are skipped and interrupted ones resume from their checkpoints, so running it again is always safe."""
import argparse, json, os, subprocess, sys, threading, time, traceback
from pathlib import Path

ap = argparse.ArgumentParser(); ap.add_argument('--plan', required=True)
plan = json.loads(Path(ap.parse_args().plan).read_text())
work = Path(plan['work']); logs = work / 'logs'; logs.mkdir(parents=True, exist_ok=True)
state_file = work / 'orchestrator.json'; lock = threading.Lock()
state = {'pid': os.getpid(), 'started': time.time(), 'status': 'running', 'done': [], 'failed': []}
def save(**kw):
    with lock:
        state.update(kw); tmp = state_file.with_suffix('.tmp'); tmp.write_text(json.dumps(state, indent=1)); os.replace(tmp, state_file)
def run(cmd, log, env=None):
    with open(log, 'a') as handle:
        handle.write(f'\n===== started {time.strftime("%H:%M:%S")} =====\n'); handle.flush()
        return subprocess.run(cmd, stdout=handle, stderr=subprocess.STDOUT, env=env).returncode
def run_status(name):
    f = work / 'runs' / name / 'run.json'
    return json.loads(f.read_text()).get('status') if f.exists() else None

def queue(gpu, jobs):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), TOKENIZERS_PARALLELISM='false')
    for job in jobs:
        name = job['name']
        for attempt in (1, 2):
            if run_status(name) == 'done': break
            code = run([sys.executable, 'worker_unsloth.py', *job['args']], logs / f'{name}.log', env)
            if code == 0 or run_status(name) != 'running': break   # retry only a run that was killed mid-way
            with open(logs / f'{name}.log', 'a') as h: h.write(f'\n===== exit code {code}: killed mid-run, resuming from the last checkpoint =====\n')
        with lock: (state['done'] if run_status(name) == 'done' else state['failed']).append(name)
        save()

try:
    save()
    py = sys.executable
    if not (work / 'train_pool.parquet').exists():
        if run([py, 'prepare.py', '--data', plan['data'], '--pool', plan['pool'], '--work', str(work), '--per-class', str(plan['per_class'])], logs / 'prepare.log'):
            save(status='failed', failed=['prepare']); sys.exit(1)
    if not (work / 'ngram.npz').exists():   # finish before the GPU runs, so the memory peaks never overlap
        if run([py, 'ngram.py', '--data', plan['data'], '--work', str(work), '--eval-rows', str(plan['eval_rows'])], logs / 'ngram.log', dict(os.environ, CUDA_VISIBLE_DEVICES='')):
            state['failed'].append('ngram')
    threads = [threading.Thread(target=queue, args=(gpu, jobs)) for gpu, jobs in plan['queues'].items()]
    for t in threads: t.start()
    for t in threads: t.join()
    report_ok = run([py, 'report.py', '--data', plan['data'], '--work', str(work)], logs / 'report.log') == 0
    save(status='finished', report_ok=report_ok)
except Exception:
    save(status='failed', error=traceback.format_exc()[-2000:]); raise


In [ ]:
%%writefile report.py
"""Compare the runs on the official rows. The test is cross-family pairs against same-family pairs of equal size."""
import argparse, json
from pathlib import Path
import numpy as np, pandas as pd

ap = argparse.ArgumentParser()
ap.add_argument('--data', required=True); ap.add_argument('--work', required=True)
a = ap.parse_args()
work = Path(a.work)
off = pd.read_csv(Path(a.data) / 'train.csv'); labels = sorted(off.model.unique()); to_id = {l: i for i, l in enumerate(labels)}
runs = {}; info = {}
for d in sorted((work / 'runs').glob('*')):
    meta = json.loads((d / 'run.json').read_text()) if (d / 'run.json').exists() else {'status': 'missing'}
    info[d.name] = meta
    if meta.get('status') == 'done' and (d / 'pred.npz').exists(): runs[d.name] = np.load(d / 'pred.npz')['logp'].astype(np.float64)
ng = np.load(work / 'ngram.npz')['logp'].astype(np.float64) if (work / 'ngram.npz').exists() else None
if not runs: raise SystemExit('No finished runs: read the worker logs above.')
n = next(iter(runs.values())).shape[0]; off = off.head(n)
y = off.model.map(to_id).to_numpy(); g = off.prompt_id.to_numpy(); prompts = np.unique(g); where = {p: np.where(g == p)[0] for p in prompts}

def score(p, rows=slice(None)):
    pred, t = p.argmax(1)[rows], y[rows]
    return float(np.mean([(pred[t == k] == k).mean() for k in range(18) if (t == k).any()]))
blend = lambda *m: sum(m) / len(m)
rng = np.random.default_rng(0); samples = [np.concatenate([where[p] for p in rng.choice(prompts, len(prompts))]) for _ in range(2000)]
def compare(tests, control):
    d = np.array([np.mean([score(t, s) for t in tests]) - score(control, s) for s in samples])
    return float(d.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5))

lines = ['## Single models', '', '| Run | Score | Rows seen | Minutes | Final loss |', '|---|---|---|---|---|']
for k, m in info.items():
    if k in runs: lines.append(f"| {k} | {score(runs[k]):.4f} | {m.get('rows_seen')} | {m.get('train_minutes')} | {m.get('final_loss')} |")
    else: lines.append(f"| {k} | FAILED: {m.get('reason', m.get('status'))} | | | |")
if ng is not None: lines.append(f'| n-gram SVC | {score(ng):.4f} | | | |')
seen = {m.get('rows_seen') for k, m in info.items() if k in runs}
if len(seen) > 1: lines += ['', f'**Note:** runs saw different numbers of rows ({sorted(seen)}), so single-model scores are not like for like.']

q, m = sorted(k for k in runs if k.startswith('qwen')), sorted(k for k in runs if k.startswith('gemma'))
verdict = 'Single-family run: there is no cross-family comparison to make.' if not (q and m) else 'Not enough finished runs for the comparison: it needs two seeds of each model.'
out = {'single': {k: score(v) for k, v in runs.items()}, 'ngram': None if ng is None else score(ng)}
if len(q) >= 2 and len(m) >= 2:
    Q1, Q2, G1, G2 = runs[q[0]], runs[q[1]], runs[m[0]], runs[m[1]]
    lines += ['', '## Pairs', '', '| Pair | Score |', '|---|---|', f'| Qwen + Qwen | {score(blend(Q1, Q2)):.4f} |', f'| Gemma + Gemma | {score(blend(G1, G2)):.4f} |',
              f'| Qwen + Gemma (first seeds) | {score(blend(Q1, G1)):.4f} |', f'| Qwen + Gemma (second seeds) | {score(blend(Q2, G2)):.4f} |']
    tests = [('Cross-family pair vs Qwen + Qwen', [blend(Q1, G1), blend(Q2, G2)], blend(Q1, Q2))]
    if ng is not None:
        tests.append(('With n-gram: cross-family vs Qwen + Qwen', [blend(ng, Q1, G1), blend(ng, Q2, G2)], blend(ng, Q1, Q2)))
    lines += ['', '## The test: cross-family minus same-family (95% interval over prompts)', '', '| Comparison | Difference | Interval |', '|---|---|---|']
    res = {}
    for name, t, c in tests:
        res[name] = compare(t, c); lines.append(f'| {name} | {res[name][0]:+.4f} | {res[name][1]:+.4f} to {res[name][2]:+.4f} |')
    key = res[tests[-1][0]]; out['tests'] = res
    if key[1] > 0: verdict = 'Gemma adds something Qwen does not: the interval is above zero. Worth keeping a Gemma member in the ensemble.'
    elif key[2] < 0: verdict = 'A second family made things worse here: the interval is below zero.'
    else: verdict = 'Inconclusive: the interval includes zero, so this run cannot separate a real gain from noise.'
prep = json.loads((work / 'prepare.json').read_text())
cover = prep.get('official_rows_on_a_prompt_the_training_rows_cover')
lines += ['', f'**Verdict:** {verdict}', '', 'Scores are on all official rows, which no model trained on. Blends use equal weights.',
          f"Overlap check: {cover:.1%} of official rows look like they sit on a prompt the training rows cover; {prep['official_rows_sharing_20pct_of_8grams']:.1%} share over 20% of their 8-grams."]
(work / 'report.md').write_text('\n'.join(lines) + '\n'); out['verdict'] = verdict; (work / 'report.json').write_text(json.dumps(out, indent=1))
print('\n'.join(lines))


## 3. Install Unsloth and check it imports

In [ ]:
import importlib.metadata, json, subprocess, sys
from pathlib import Path
def installed(package):
    try: return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: return None
def snapshot(): return {(d.metadata['Name'] or '').lower(): d.version for d in importlib.metadata.distributions()}
if installed('unsloth') != UNSLOTH_VERSION:
    before = snapshot()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'unsloth=={UNSLOTH_VERSION}', f'unsloth_zoo=={UNSLOTH_VERSION}'], check=True)
    added = {k: v for k, v in snapshot().items() if k and before.get(k) != v}
    Path('/kaggle/working/offline').mkdir(parents=True, exist_ok=True)
    Path('/kaggle/working/offline/installed_by_notebook.json').write_text(json.dumps(added, indent=1))   # used by the last cell
    print(f'install added or changed {len(added)} packages')
check = subprocess.run([sys.executable, 'check_env.py'], capture_output=True, text=True)
print('\n'.join(l for l in check.stdout.splitlines() if not l.startswith(('🦥', '==((', '   \\', 'O^O', '\\ ', ' "-'))))
if check.returncode:
    raise SystemExit('Unsloth did not import cleanly:\n' + check.stderr[-3000:])


## 4. Download the models

In [ ]:
import json, shutil, urllib.request
from pathlib import Path

def fetch(repo):
    # Download over plain HTTP and check each file's size, so the runs need no network.
    target = Path('/tmp/models') / repo.replace('/', '--'); target.mkdir(parents=True, exist_ok=True)
    info = json.load(urllib.request.urlopen(f'https://huggingface.co/api/models/{repo}?blobs=true', timeout=60))
    for item in info['siblings']:
        name, size = item['rfilename'], item.get('size')
        if '/' in name or not name.endswith(('.json', '.safetensors', '.model', '.txt', '.jinja')): continue
        path = target / name
        if path.exists() and path.stat().st_size == size: continue
        with urllib.request.urlopen(f'https://huggingface.co/{repo}/resolve/main/{name}', timeout=120) as source, open(path, 'wb') as sink:
            shutil.copyfileobj(source, sink, 1 << 24)
        assert size is None or path.stat().st_size == size, f'{name} is incomplete'
    return str(target)

PATHS = {}
for family, repo in FAMILIES.items():
    try:
        PATHS[family] = fetch(repo)
    except Exception as error:
        raise SystemExit(f'Could not download {repo}. Is Internet switched on in the notebook settings? {error!r}')
    print('ready:', repo)


## 5. Run, or reattach to a run in progress
Safe to run again at any time: finished runs are skipped and interrupted ones resume.

In [ ]:
import glob, os, time
WORK = Path('/kaggle/working/run_smoke' if SMOKE else '/kaggle/working/run'); WORK.mkdir(parents=True, exist_ok=True)
SETTINGS = {'families': FAMILIES, 'seeds': SEEDS, 'per_class': PER_CLASS, 'limit_steps': LIMIT_STEPS, 'eval_rows': EVAL_ROWS, 'split_rows': SPLIT_ROWS}
if (WORK / 'settings.json').exists() and json.loads((WORK / 'settings.json').read_text()) != SETTINGS:   # never resume or report on a different setup
    raise SystemExit(f'{WORK} holds a run with different settings. To start over, run this in a cell: !rm -rf {WORK}')
(WORK / 'settings.json').write_text(json.dumps(SETTINGS, indent=1))
STATE = WORK / 'orchestrator.json'

def alive(pid):
    try:
        with open(f'/proc/{pid}/status') as status:
            return not any(line.startswith('State:') and 'Z' in line.split()[1] for line in status)
    except (OSError, TypeError):
        return False

state = json.loads(STATE.read_text()) if STATE.exists() else {}
if state.get('status') == 'running' and alive(state.get('pid')):
    print(f"Reattaching to the run already in progress (process {state['pid']}).")
else:
    if state.get('status') == 'running':
        print('The previous attempt stopped unexpectedly. Starting again: finished runs are kept and the rest resume from their checkpoints.')
    DATA = next((p for p in ['/kaggle/input/competitions/name-that-llm', '/kaggle/input/name-that-llm'] if os.path.exists(p + '/train.csv')), None)
    assert DATA, 'Add the Name That LLM competition data as an input.'
    pools = sorted(glob.glob('/kaggle/input/**/ntl_open_pool.parquet', recursive=True))
    assert pools, 'Add the Name That LLM Open Pool dataset as an input.'
    gpus = [l for l in subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.splitlines() if l.startswith('GPU')]
    assert gpus, 'No GPU found: choose GPU T4 x2 in the notebook settings.'
    deadline = time.time() + TOTAL_HOURS * 3600
    queues = {str(i): [] for i in range(len(gpus))}
    for n, (family, seed) in enumerate((f, sd) for f in FAMILIES for sd in SEEDS):   # spread the runs over the GPUs
        name = f'{family}_s{seed}'
        queues[str(n % len(gpus))].append({'name': name, 'args': [
            '--name', name, '--repo', PATHS[family], '--seed', str(seed), '--data', DATA, '--work', str(WORK),
            '--shard-index', str(SEEDS.index(seed) if SPLIT_ROWS else 0), '--shard-count', str(len(SEEDS) if SPLIT_ROWS else 1),
                '--budget-min', str(BUDGET_MIN), '--deadline', str(deadline), '--limit-steps', str(LIMIT_STEPS), '--eval-rows', str(EVAL_ROWS)]})
    plan = {'work': str(WORK), 'data': DATA, 'pool': pools[0], 'per_class': PER_CLASS, 'eval_rows': EVAL_ROWS, 'queues': queues}
    (WORK / 'plan.json').write_text(json.dumps(plan, indent=1))
    subprocess.Popen([sys.executable, 'orchestrate.py', '--plan', str(WORK / 'plan.json')], start_new_session=True,
                     stdout=open(WORK / 'orchestrator.log', 'a'), stderr=subprocess.STDOUT)
    for _ in range(60):
        time.sleep(2)
        if STATE.exists() and json.loads(STATE.read_text()).get('pid'): break
    print('Started. Progress appears below every few minutes.')

def free_ram():
    try: return next(int(l.split()[1]) for l in open('/proc/meminfo') if l.startswith('MemAvailable')) / 2**20
    except (OSError, StopIteration): return float('nan')

while True:
    state = json.loads(STATE.read_text())
    if state.get('status') != 'running' or not alive(state.get('pid')): break
    time.sleep(30 if SMOKE else 300)
    lines = []
    for log in sorted((WORK / 'logs').glob('*.log')):
        last = [l for l in log.read_text(errors='replace').splitlines() if l.startswith('[')]
        if last: lines.append(f'{log.stem:9s} | {last[-1][:130]}')
    print(time.strftime('%H:%M'), f'RAM free {free_ram():.1f}GB', *lines, sep='\n  ', flush=True)


## 6. Result

In [ ]:
state = json.loads(STATE.read_text())
for name in state.get('failed', []):
    log = WORK / 'logs' / f'{name}.log'
    if log.exists(): print(f'--- {name} failed ---', *log.read_text(errors='replace').strip().splitlines()[-15:], sep='\n')
if state.get('status') == 'running':
    print('The orchestrator stopped without finishing (the kernel or VM may have restarted). Run the last two cells again to resume.')
elif state.get('status') == 'failed':
    print('The orchestrator failed:', state.get('error', '')[-1500:])
if (WORK / 'report.md').exists():
    print('SMOKE TEST ONLY: these numbers mean nothing. Set SMOKE = False for the real run.\n' if SMOKE else '')
    print((WORK / 'report.md').read_text())


## 7. Save what an offline submission notebook needs
Scoring has no internet. This copies the base model and the packages the install step added into `/kaggle/working/offline`. The results above are already saved, so a failure here loses nothing.


In [ ]:
import json, shutil, subprocess, sys
from pathlib import Path
OFFLINE = Path('/kaggle/working/offline'); OFFLINE.mkdir(parents=True, exist_ok=True)
for family in FAMILIES:
    try:
        dest = OFFLINE / f'base_{family}'
        if not dest.exists(): shutil.copytree(PATHS[family], dest)
        print(f'base model saved: {dest} ({sum(f.stat().st_size for f in dest.rglob("*") if f.is_file()) / 2**30:.1f} GB)')
    except Exception as error:
        print(f'Could not copy the {family} base model: {error!r}')
try:
    listed = OFFLINE / 'installed_by_notebook.json'
    added = json.loads(listed.read_text()) if listed.exists() else {}
    wheels = OFFLINE / 'wheels'; wheels.mkdir(exist_ok=True)
    for name, version in added.items():
        subprocess.run([sys.executable, '-m', 'pip', 'download', '-q', '--no-deps', '-d', str(wheels), f'{name}=={version}'], timeout=900)
    saved = sorted(p.name for p in wheels.iterdir())
    print(f'{len(saved)} package files saved for {len(added)} packages the install added')
    missing = [n for n in added if not any(f.lower().replace('_', '-').startswith(n.replace('_', '-') + '-') for f in saved)]
    if missing: print('Not saved (check these before relying on offline install):', missing)
except Exception as error:
    print(f'Could not save the packages: {error!r}')
